In [2]:
# Setting up the Environment
# We installed these specific versions for compatibility reasons (The file was made in DeepNote)

%pip install numpy==1.26.4 pandas scikit-learn==1.5.0 dice-ml --user

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import dice_ml
import numpy as np

In [22]:
df = pd.read_csv("adult.csv")
df.replace("?", np.nan, inplace=True) # We replace the ? mark with a NaN object, and then remove the rows containing them.
df = df.dropna()
df = df.drop(columns=['fnlwgt']) # We remove the 'fnlwgt' column since it not an actual characteristic of the candidates.

df.head()


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K


In [23]:
# We make our target 'income' and convert it to a binary value (0 and 1) just like we did in SHAP.
target = 'income'
df[target] = df[target].map({'<=50K': 0, '>50K': 1})

# DiCE Needs to know which featuers are numerical and which are categorical.
# It needs this because to suggest changes to cross the decision boundary, it needs to know how these variables can be changed.

numerical_features = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']

In [24]:
# We define our input and our target for the model.
X = df.drop(target, axis=1)
y = df[target]

# We split the data into a 80% Training set, and 20% Test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [25]:
# Building the Pipeline and Training

numeric_transformer = StandardScaler() # For the numeric variables we use a Standard scaler, which scales features like 'age'.
categorical_transformer = OneHotEncoder(handle_unknown='ignore') # We use a OneHotEncoder to label the categorical features into binary values.

# Here we transform both these features using the Scaler and OneHotEncoder defined above.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Here we create a Pipeline using the preprocessor and the random forest
# This ensures that even raw data first gets preprocessed.
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# We then train the model and check for the accuracy.
clf.fit(X_train, y_train)
print(f"Model Accuracy: {clf.score(X_test, y_test):.2%}")

Model Accuracy: 84.48%


In [33]:

# Here, we pass the training data and label the continous (numerical) features, this tells DiCE it how much it can change the numerical features, and the options allowed for categorical changes.
d = dice_ml.Data(dataframe=pd.concat([X_train, y_train], axis=1),
    continuous_features=numerical_features,
    outcome_name=target)

# The Model object now looks at the entire pipeline, including the scaler and OneHotEncoder.
m = dice_ml.Model(model=clf, backend="sklearn")

# We let DiCE search for Counterfactuals. The reason we use "genetic" is because this ensures that DiCE is not guessing randomly.
# It chooses the scenarios that require the least amount of change in order for the decision to change.
exp = dice_ml.Dice(d, m, method="genetic")

# We pick a person from the test set, who was predicted to have a Low income. (0)
query_instance = X_test[clf.predict(X_test) == 0].iloc[6:7] # You could replace this with .sample to find any random amount of candidates.

# We ask DiCE to find ways (here: 20) this candidate would reach a High income output by changing the features in the dataset.
# Then we Visualise this, showing what features were changed to reach the output.
dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=20, desired_class=1)
dice_exp.visualize_as_dataframe(show_only_changes=True)

  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\peter\AppData\Roaming\Python\Python311\site-packages\dice_ml\explainer_interfaces\dice_genetic.py:285: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  query_instance_df_dummies[col] = 0
100%|██████████| 1/1 [00:03<00:00,  3.26s/it]

Query instance (original outcome : 0)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,46,Local-gov,Assoc-voc,11,Divorced,Exec-managerial,Not-in-family,White,Female,0,0,36,United-States,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,-,-,-,-,Married-civ-spouse,-,Husband,-,Male,-,-,40,-,1
0,45,Private,-,-,-,-,-,-,Male,-,-,40,-,1
0,43,Private,-,-,Married-civ-spouse,Prof-specialty,Wife,-,-,-,-,-,-,1
0,-,Federal-gov,-,-,Married-civ-spouse,-,Husband,-,Male,-,-,40,-,1
0,-,Private,-,-,Married-civ-spouse,-,Husband,-,Male,-,-,40,-,1
0,-,Self-emp-not-inc,-,-,Married-civ-spouse,-,Husband,-,Male,-,-,40,-,1
0,-,Private,-,-,Married-civ-spouse,Prof-specialty,Wife,-,-,-,-,40,-,1
0,45,Private,-,-,Married-civ-spouse,-,Wife,-,-,-,-,40,-,1
0,47,-,Bachelors,13,-,Prof-specialty,-,-,-,-,-,35,-,1
0,-,Private,HS-grad,9,-,-,Unmarried,-,-,-,-,38,-,1


In [34]:
# To find biases and reduce the Rashomon Effect, here we lock the only features that are allowed to change, to the personal features.
# If DiCE can still find ways to change these features, which will predict High income instead of Low, we have found a bias in our model.
locked_features_to_vary = [
    'age', 'marital-status', 'relationship', 'race', 'gender', 
]

try:
    # We try to find Counterfactuals (Here: 5) where we only change personal features of the candidate in our Query instance.
    dice_exp = exp.generate_counterfactuals(
        query_instance, 
        total_CFs=3, 
        desired_class=1,
        features_to_vary=locked_features_to_vary # This forces Occupation to stay the same
    )
    # We visualise this in the same way if DiCE found a Counterfactual.
    dice_exp.visualize_as_dataframe(show_only_changes=True)
except Exception as e:
    # Print this in case the model was unable to find any counterfactuals, implying the model predicts based on professional features.
    print("The model could not find a way to hire this person without changing their occupation.")


  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\peter\AppData\Roaming\Python\Python311\site-packages\dice_ml\explainer_interfaces\dice_genetic.py:285: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  query_instance_df_dummies[col] = 0
100%|██████████| 1/1 [00:17<00:00, 17.55s/it]

Query instance (original outcome : 0)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,46,Local-gov,Assoc-voc,11,Divorced,Exec-managerial,Not-in-family,White,Female,0,0,36,United-States,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,47,-,-,-,Married-civ-spouse,-,Wife,-,-,-,-,-,-,1
0,44,-,-,-,Married-civ-spouse,-,Wife,-,-,-,-,-,-,1
0,43,-,-,-,Married-civ-spouse,-,Wife,-,-,-,-,-,-,1


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2a4fae61-7c6d-4f44-aaa3-bddc3bc318f1' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>